In [ ]:
import json
from pathlib import Path
from typing import Dict, List, Literal, Tuple, cast

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_recall_curve, auc


def minimum(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    return dq_results.min(axis=0)


def maximum(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    return dq_results.max(axis=0)


def weighted_minimum(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    weighted_results = dq_results * certainties
    return weighted_results.min(axis=0)


def weighted_maximum(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    weighted_results = dq_results * certainties
    return weighted_results.max(axis=0)


def mean(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    return dq_results.mean(axis=0)


def weighted_mean(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    weighted_sums = (dq_results * certainties).sum(axis=0)
    sum_of_weights = certainties.sum(axis=0)
    return weighted_sums / sum_of_weights


def median(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    return dq_results.median(axis=0)


def weighted_median(dq_results: pd.DataFrame, certainties: pd.DataFrame) -> pd.Series:
    weighted_results = dq_results * certainties
    return weighted_results.median(axis=0)


def mse_per_column(expected: pd.DataFrame, predicted: pd.DataFrame) -> pd.Series:
    errors = (expected - predicted) ** 2
    return errors.mean()


def evaluate_aggregation_methods(
    dq_results: pd.DataFrame, certainties: pd.DataFrame, expected: pd.DataFrame
) -> pd.DataFrame:
    aggregation_methods = [
        minimum,
        maximum,
        weighted_minimum,
        weighted_maximum,
        mean,
        weighted_mean,
        median,
        weighted_median,
    ]

    mse_results = {}
    for method in aggregation_methods:
        aggregated_results = method(dq_results, certainties)
        mse = ((expected - aggregated_results) ** 2).mean()
        mse_results[method.__name__] = mse

    return pd.DataFrame(mse_results)


def parse_columnNames(columnNames_str: str) -> str:
    columnNames = json.loads(str(columnNames_str).replace("'", '"'))
    return columnNames[0] if len(columnNames) == 1 else ",".join(columnNames)


def evaluate_run(run: str):
    folder = Path(f"results/{run}")
    data_configs: List[Tuple[Path, str]] = [
        (file, file.name.replace(".loader_config.json", ""))
        for file in folder.rglob("*.loader_config.json")
    ]

    reference_per_data_config: Dict[
        str,
        Dict[Literal["data", "mask"], pd.DataFrame],
    ] = {}

    for path, name in data_configs:
        with open(path, "r") as f:
            data_config = json.load(f)
        data = pd.read_csv(data_config["file_name"])
        mask_file = data_config["file_name"].replace(".csv", ".mask.csv")
        if not Path(mask_file).exists():
            print(f"No mask found for {name}")
            mask = pd.DataFrame(False, data.index, data.columns)
        else:
            mask = pd.read_csv(mask_file)
        reference_per_data_config[name] = {
            "data": data,
            "mask": mask,
        }

    dq_results_flat = pd.read_csv(folder / "dq_results.csv")

    dq_results_per_metric_and_dataset: Dict[
        str,
        Dict[str, pd.DataFrame],
    ] = {}
    certainties_per_metric_and_dataset: Dict[
        str,
        Dict[str, pd.DataFrame],
    ] = {}

    for key, index in dq_results_flat.groupby(
        ["tableName", "DQmetric", "columnNames"]
    ).groups.items():
        tableName, DQmetric, columnNames = cast(tuple, key)
        column = parse_columnNames(str(columnNames))
        metric = str(DQmetric)
        dataset = str(tableName)
        group = dq_results_flat.loc[index]
        if group["rowIndex"].duplicated().any():
            raise ValueError(
                f"Duplicate rowIndex found for table {tableName}, metric {DQmetric}, column {columnNames}"
            )

        dq_results_per_metric_and_dataset.setdefault(metric, {})
        if dataset not in dq_results_per_metric_and_dataset[metric]:
            polluted_data = reference_per_data_config[dataset]["data"]
            dq_results_per_metric_and_dataset[metric][dataset] = pd.DataFrame(
                None, index=polluted_data.index, columns=polluted_data.columns
            )
        dq_results_per_metric_and_dataset[metric][dataset].loc[
            group["rowIndex"].tolist(), column
        ] = group["DQvalue"].to_numpy()

        certainties = group["DQannotations"].apply(
            lambda x: json.loads(str(x).replace("'", '"'))["certainty"] if x and not pd.isna(x) and len(x) > 0 else 1.0
        )

        certainties_per_metric_and_dataset.setdefault(metric, {})
        if dataset not in certainties_per_metric_and_dataset[metric]:
            polluted_data = reference_per_data_config[dataset]["data"]
            certainties_per_metric_and_dataset[metric][dataset] = pd.DataFrame(
                None, index=polluted_data.index, columns=polluted_data.columns
            )
        certainties_per_metric_and_dataset[metric][dataset].loc[
            group["rowIndex"].tolist(), column
        ] = certainties.to_numpy()

    # Remove columns without data
    for per_dataset in dq_results_per_metric_and_dataset.values():
        for dataframe in per_dataset.values():
            dataframe.dropna(axis=1, inplace=True, how="all")
    for per_dataset in certainties_per_metric_and_dataset.values():
        for dataframe in per_dataset.values():
            dataframe.dropna(axis=1, inplace=True, how="all")

    for (dq_metric, dq_results_per_dataset), certainties_per_dataset in zip(
        dq_results_per_metric_and_dataset.items(),
        certainties_per_metric_and_dataset.values(),
    ):
        print(f"# Evaluating DQ metric: {dq_metric}")
        for (dataset, dq_results), certainties in zip(
            dq_results_per_dataset.items(), certainties_per_dataset.values()
        ):
            print(f"Dataset: {dataset}")
            print("\nDQ results")
            print(dq_results.describe())
            print("\nCertainties")
            print(certainties.describe())
            print(certainties.quantile([0.0, 0.25, 0.5, 0.75, 1.0]))
            if any("," in col for col in dq_results.columns.to_list()):
                print("Evaluating tuple base aggregation")
                mask = ~reference_per_data_config[dataset]["mask"]
                mask = pd.DataFrame(mask.mean(axis=1), columns=dq_results.columns)
            else:
                # mask has to be inverted because True indicates polluted values for which the quality should be 0
                mask = ~reference_per_data_config[dataset]["mask"][
                    dq_results.columns.tolist()
                ]

            print("\nMask")
            print(mask.mean())

            mse_per_column_values_no_quality = mse_per_column(
                mask,
                pd.DataFrame(1.0, index=dq_results.index, columns=dq_results.columns),
            )
            mse_per_column_values = mse_per_column(mask, dq_results)
            mse_per_column_values_with_certainty = mse_per_column(
                mask, dq_results * certainties
            )
            results = {
                "Without quality values": mse_per_column_values_no_quality,
                "With quality values": mse_per_column_values,
                "With certainty weighting": mse_per_column_values_with_certainty,
            }

            print("\nMSE per column:")
            print(pd.DataFrame(results))
            print("\nDQ results value counts:")
            print(dq_results.value_counts())
            print()

            # print(evaluate_aggregation_methods(dq_results, certainties, mask))

            # plot_accuracy_by_threshold(dq_results, ~mask, dq_results.columns.tolist())
            # plot_f1_score_by_threshold(dq_results, ~mask, dq_results.columns.tolist())
            # plot_pr_auc_curve(dq_results, ~mask, dq_results.columns.tolist())

    return dq_results_per_metric_and_dataset


def plot_accuracy_by_threshold(
    dq_results: pd.DataFrame, mask: pd.DataFrame, columns: List[str]
):
    # Accuracy (correctly classified observations) should not be used on imbalanced data
    thresholds = [i / 100 for i in range(0, 101)]
    accuracies_per_column = {}
    for threshold in thresholds:
        accuracy = ((dq_results[columns] >= threshold) == (~mask[columns])).mean()
        for col in accuracy.index:
            accuracies_per_column.setdefault(col, []).append(accuracy[col])

    fig, ax = plt.subplots()

    for col, accuracies in accuracies_per_column.items():
        ax.plot(thresholds, accuracies, label=col)
    ax.set_xlabel("DQ threshold")
    ax.set_ylabel("Accuracy")
    ax.set_title("DQ Accuracy by Threshold")
    ax.grid()
    ax.legend()
    plt.show(fig)


def plot_f1_score_by_threshold(
    dq_results: pd.DataFrame, mask: pd.DataFrame, columns: List[str]
):
    thresholds = [i / 100 for i in range(0, 101)]
    f1_scores_per_column = {}
    for threshold in thresholds:
        predictions = dq_results[columns] >= threshold
        for col in columns:
            f1 = f1_score(~mask[col], predictions[col])
            f1_scores_per_column.setdefault(col, []).append(f1)

    fig, ax = plt.subplots()

    for col, f1_scores in f1_scores_per_column.items():
        ax.plot(thresholds, f1_scores, label=col)
    ax.set_xlabel("DQ threshold")
    ax.set_ylabel("F1 Score")
    ax.set_title("DQ F1 Score by Threshold")
    ax.grid()
    ax.legend()
    plt.show(fig)


def plot_pr_auc_curve(dq_results: pd.DataFrame, mask: pd.DataFrame, columns: List[str]):
    fig, ax = plt.subplots()

    for col in columns:
        precision, recall, t = precision_recall_curve(
            ~mask[col][dq_results[col].dropna().index], dq_results[col].dropna()
        )
        pr_auc = auc(recall, precision)
        ax.plot(recall, precision, label=f"{col} (AUC={pr_auc:.2f})")

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve")
    ax.grid()
    ax.legend()
    plt.show(fig)


with pd.option_context('display.expand_frame_repr', False):
    # evaluate_run("20260127_231441")
    # Completeness
    evaluate_run("20260131_165047")
    # evaluate_run("20260115_073708_consistency")
    # evaluate_run("20260115_073742_timeliness")

None

No mask found for weather
No mask found for movies
No mask found for auto_sales


/var/folders/rw/xym_kq6n3l9_0qkvb5l3bd_w0000gn/T/ipykernel_13983/2832718090.py:109: DtypeWarning: Columns (0: DQannotations) have mixed types. Specify dtype option on import or set low_memory=False.
  dq_results_flat = pd.read_csv(folder / "dq_results.csv")


# Evaluating DQ metric: completeness_nullAndDMVRate
Dataset: auto_sales

DQ results
       ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,DAYS_SINCE_LASTORDER,STATUS,PRODUCTLINE,MSRP,PRODUCTCODE,CUSTOMERNAME,PHONE,ADDRESSLINE1,CITY,POSTALCODE,COUNTRY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
count                                        2747.000000                                                                                                                                                                               
mean                                            0.936403                                                                                                                                                                               
std                                             0.098864                                                                                                                                                                    